#CELL 0: ENVIRONMENT SETUP

In [1]:
import torch
import os

print("Setting up PyTorch Geometric for Colab...")

# 1. Get the current PyTorch and CUDA versions
torch_version = torch.__version__.split('+')[0]
cuda_version = torch.version.cuda.replace('.', '')

# 2. Build the URL to the pre-compiled files
wheel_url = f"https://data.pyg.org/whl/torch-{torch_version}+cu{cuda_version}.html"

# 3. Install PyG and its dependencies
os.system(f"pip install torch_geometric")
os.system(f"pip install pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv -f {wheel_url}")

print("✅ Installation Complete! You can now run Cell 1.")

Setting up PyTorch Geometric for Colab...
✅ Installation Complete! You can now run Cell 1.


# CELL 1: IMPORTS & SETUP (MULTI-OMICS)

In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch_geometric.data import Data
from sklearn.neighbors import kneighbors_graph
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.model_selection import StratifiedKFold

print("✅ Cell 1: Multi-Omics Libraries imported successfully!")

✅ Cell 1: Multi-Omics Libraries imported successfully!


# CELL 2: LOAD, ALIGN & EXTRACT 3 MODALITIES

In [5]:
# CELL 2: LOAD & EXTRACT 3 MODALITIES (Pre-Aligned Data)
print("Loading all 3 TCGA Datasets from GitHub...")

url_mrna = 'https://raw.githubusercontent.com/shawkath73/GCN-implementation/refs/heads/main/data/tcga/BRCA%20mRNA%20SUBTYPE.csv'
url_methy = 'https://raw.githubusercontent.com/shawkath73/GCN-implementation/refs/heads/main/data/tcga/BRCA%20METHYL%20SUBTYPE.csv'
url_mirna = 'https://raw.githubusercontent.com/shawkath73/GCN-implementation/refs/heads/main/data/tcga/BRCA%20MIRNA%20SUBTYPE.csv'

df_mrna = pd.read_csv(url_mrna)
df_methy = pd.read_csv(url_methy)
df_mirna = pd.read_csv(url_mirna)

# 1. Clean TCGA column names (Removes the |12345 suffix)
df_mrna.columns = [col.split('|')[0] if '|' in col else col for col in df_mrna.columns]
df_methy.columns = [col.split('|')[0] if '|' in col else col for col in df_methy.columns]
df_mirna.columns = [col.split('|')[0] if '|' in col else col for col in df_mirna.columns]

# 2. Encode Labels (Using mRNA labels as the master key)
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(df_mrna['Label'].values)

# 3. Extract Features (Since data is pre-aligned, we only drop the 'Label' column)
features_mrna = [col for col in df_mrna.columns if col != 'Label']
features_methy = [col for col in df_methy.columns if col != 'Label']
features_mirna = [col for col in df_mirna.columns if col != 'Label']

X_mrna_raw = df_mrna[features_mrna].values
X_methy_raw = df_methy[features_methy].values
X_mirna_raw = df_mirna[features_mirna].values

# 4. Variance Filtering (As per the paper: std < 0.1 means variance < 0.01)
selector = VarianceThreshold(threshold=0.01)
X_mrna_filtered = selector.fit_transform(X_mrna_raw)
X_methy_filtered = selector.fit_transform(X_methy_raw)
X_mirna_filtered = selector.fit_transform(X_mirna_raw)

print(f"✅ Cell 2 Complete! Data extracted and filtered.")
print(f"Total Patients: {len(y_encoded)}")
print(f"Target Classes: {list(label_encoder.classes_)}")
print(f"mRNA Features: {X_mrna_filtered.shape[1]}")
print(f"Methylation Features: {X_methy_filtered.shape[1]}")
print(f"miRNA Features: {X_mirna_filtered.shape[1]}")

Loading all 3 TCGA Datasets from GitHub...
✅ Cell 2 Complete! Data extracted and filtered.
Total Patients: 875
Target Classes: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
mRNA Features: 278
Methylation Features: 335
miRNA Features: 15


# CELL 3: GRAPH CONSTRUCTION

In [6]:
def build_graph(X, y, k_neighbors=5):
    # Scale data independently for this specific modality and fold
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    x_tensor = torch.tensor(X_scaled, dtype=torch.float)
    y_tensor = torch.tensor(y, dtype=torch.long)

    # Build K-Nearest Neighbors Adjacency Matrix for this specific modality
    knn_matrix = kneighbors_graph(X_scaled, n_neighbors=k_neighbors, mode='connectivity', include_self=False)
    edge_index = torch.tensor(np.array(knn_matrix.nonzero()), dtype=torch.long)

    return Data(x=x_tensor, edge_index=edge_index, y=y_tensor)

print("✅ Cell 3: Graph construction function ready!")

✅ Cell 3: Graph construction function ready!


# CELL 4: MULTI-OMICS FUSION ARCHITECTURE

In [7]:
class MultiOmicsGCN(torch.nn.Module):
    def __init__(self, num_feat_mrna, num_feat_methy, num_feat_mirna, hidden_1=64, hidden_2=32, num_classes=5):
        super(MultiOmicsGCN, self).__init__()

        # Branch 1: mRNA
        self.gcn_mrna1 = GCNConv(num_feat_mrna, hidden_1)
        self.gcn_mrna2 = GCNConv(hidden_1, hidden_2)

        # Branch 2: DNA Methylation
        self.gcn_methy1 = GCNConv(num_feat_methy, hidden_1)
        self.gcn_methy2 = GCNConv(hidden_1, hidden_2)

        # Branch 3: miRNA
        self.gcn_mirna1 = GCNConv(num_feat_mirna, hidden_1)
        self.gcn_mirna2 = GCNConv(hidden_1, hidden_2)

        # LATE FUSION LAYER: 32 (mRNA) + 32 (Methy) + 32 (miRNA) = 96 incoming features
        self.classifier = torch.nn.Linear(hidden_2 * 3, num_classes)

    def forward(self, data_mrna, data_methy, data_mirna):
        # Process mRNA
        x_mrna = F.relu(self.gcn_mrna1(data_mrna.x, data_mrna.edge_index))
        x_mrna = F.dropout(x_mrna, p=0.5, training=self.training)
        x_mrna = F.relu(self.gcn_mrna2(x_mrna, data_mrna.edge_index))

        # Process Methylation
        x_methy = F.relu(self.gcn_methy1(data_methy.x, data_methy.edge_index))
        x_methy = F.dropout(x_methy, p=0.5, training=self.training)
        x_methy = F.relu(self.gcn_methy2(x_methy, data_methy.edge_index))

        # Process miRNA
        x_mirna = F.relu(self.gcn_mirna1(data_mirna.x, data_mirna.edge_index))
        x_mirna = F.dropout(x_mirna, p=0.5, training=self.training)
        x_mirna = F.relu(self.gcn_mirna2(x_mirna, data_mirna.edge_index))

        # FUSE ALL THREE VIEWS
        fused = torch.cat([x_mrna, x_methy, x_mirna], dim=1)

        out = self.classifier(fused)
        return F.log_softmax(out, dim=1)

print("✅ Cell 4: Multi-Omics Fusion Network defined!")

✅ Cell 4: Multi-Omics Fusion Network defined!


# CELL 5: 5-FOLD CROSS-VALIDATION PIPELINE (MULTI-OMICS)

In [11]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_accuracies = []

print("Starting Multi-Omics 5-Fold Cross-Validation...\n" + "="*50)

for fold, (train_idx, test_idx) in enumerate(skf.split(X_mrna_filtered, y_encoded)):

    # 1. Split data for all 3 modalities
    # mRNA
    mrna_train, mrna_test = X_mrna_filtered[train_idx], X_mrna_filtered[test_idx]
    # Methylation
    methy_train, methy_test = X_methy_filtered[train_idx], X_methy_filtered[test_idx]
    # miRNA
    mirna_train, mirna_test = X_mirna_filtered[train_idx], X_mirna_filtered[test_idx]

    y_train, y_test = y_encoded[train_idx], y_encoded[test_idx]

    # 2. Build isolated graphs for all 3 modalities (This converts labels to Tensors!)
    graph_mrna_train = build_graph(mrna_train, y_train)
    graph_mrna_test = build_graph(mrna_test, y_test)

    graph_methy_train = build_graph(methy_train, y_train)
    graph_methy_test = build_graph(methy_test, y_test)

    graph_mirna_train = build_graph(mirna_train, y_train)
    graph_mirna_test = build_graph(mirna_test, y_test)

    # 3. Initialize Multi-Omics Model
    model = MultiOmicsGCN(
        num_feat_mrna=graph_mrna_train.num_node_features,
        num_feat_methy=graph_methy_train.num_node_features,
        num_feat_mirna=graph_mirna_train.num_node_features
    )
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

    # 4. Training Loop
    model.train()
    for epoch in range(100):
        optimizer.zero_grad()
        out = model(graph_mrna_train, graph_methy_train, graph_mirna_train)

        # 👉 THE FIX: Pull the tensor labels directly from the graph object
        loss = F.nll_loss(out, graph_mrna_train.y)

        loss.backward()
        optimizer.step()

    # 5. Evaluation Phase
    model.eval()
    with torch.no_grad():
        pred = model(graph_mrna_test, graph_methy_test, graph_mirna_test).argmax(dim=1)

        # 👉 THE FIX: Compare predictions against the tensor labels in the graph
        correct = (pred == graph_mrna_test.y).sum().item()
        acc = correct / len(graph_mrna_test.y)
        fold_accuracies.append(acc)

        print(f"Fold {fold + 1} | Test Accuracy: {acc * 100:.2f}%")

# 6. Final Results
final_accuracy = np.mean(fold_accuracies) * 100
print("="*50)
print(f"🏆 FINAL MULTI-OMICS 5-FOLD CV ACCURACY: {final_accuracy:.2f}%")
print("="*50)

Starting Multi-Omics 5-Fold Cross-Validation...
Fold 1 | Test Accuracy: 76.57%
Fold 2 | Test Accuracy: 76.57%
Fold 3 | Test Accuracy: 72.00%
Fold 4 | Test Accuracy: 79.43%
Fold 5 | Test Accuracy: 78.29%
🏆 FINAL MULTI-OMICS 5-FOLD CV ACCURACY: 76.57%
